# Meta-Brain M0/M1: conectoma completo, controles e propagação esparsa treinável

Este notebook é **autocontido**: não clona o repositório. O código de `meta_brain/graph.py`, `shuffle.py`, `sparse_ops.py` e `benchmark.py` está embutido nas células da seção 2. O arquivo é gerado por `scripts/build_colab_m0_m1_notebook.py`, então não o edite à mão.

O que ele faz:

1. Baixa os três arquivos oficiais do MaleCNS v1.0 (anotações, neurotransmissores e pesos, ~1,2 GB) ou os reutiliza do Google Drive.
2. **M0, carga do grafo:** monta o conectoma de neurônios `Traced` (descartando as linhas que ligam a fragmentos não rastreados), faz cache em disco e contabiliza cada linha lida.
3. **M0, controles:** gera os três nulos (preserva grau, preserva tipo, aleatório) e verifica as invariantes de cada um.
4. **M1, correção:** roda `gradcheck` e compara o operador customizado com a referência por autograd no grafo completo.
5. **M1, benchmark:** mede BPTT pelo cérebro inteiro (K micro-passos por frame, checkpoint por frame) e compara tempo e memória com a referência.

Ao final, salva gráficos PNG, um CSV com o benchmark e `m0_m1_results.json` em `/content/meta_brain_m0_m1`, além de um `.zip` com tudo.

**Critérios** (`claim_status` no JSON):
- **M0:** todas as linhas de aresta contabilizadas, guard de escala, cache carregando em menos de 2 min e controles exatos (resíduo < 0,1%).
- **M1:** `gradcheck` e equivalência com a referência; `K=8`, `B=8` e janela de 16 frames cabem em 16 GB de GPU.

Sem GPU CUDA, o M1 fica `NOT_EVALUATED_NO_CUDA`. O critério não é avaliado em CPU.

In [ ]:
# Colab already ships torch, numpy, pandas and matplotlib.
# No -U: upgrading pyarrow/requests breaks packages Colab pins (google-colab, cudf).
!pip -q install pyarrow psutil requests

In [ ]:
import gc, json, os, platform, shutil, sys, time, zipfile
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def _env(name, default, cast=str):
    value = os.environ.get(name)
    return default if value in (None, "") else cast(value)


IN_COLAB = "google.colab" in sys.modules
SOURCE_DIGEST = "7f044d38e678"  # sha1 of the embedded meta_brain/*.py sources

# --- Parameters ---------------------------------------------------------------
# True: cache the 1.2 GB download and the graph cache on Google Drive (asks for Drive access).
USE_DRIVE = False  # @param {type:"boolean"}
STATUSES = ("Traced",)  # bodies kept from the annotations table
MIN_WEIGHT = 1.0  # minimum synapses per neuron pair
SHUFFLE_SEED = 0
MIN_NEURONS, MIN_EDGES = 100_000, 5_000_000  # full-brain guard
M0_CACHE_LOAD_TARGET_S = 120.0  # M0: the cached graph loads in under 2 minutes
M0_SHUFFLE_RESIDUAL_MAX = 1e-3  # M0: at most 0.1% of edges merged/dropped by a control
M1_MEMORY_TARGET_GIB = 16.0  # M1: K=8, B=8, 16-frame BPTT window fits in 16 GB
MICRO_STEPS = 8
BENCH = "full"  # "full" on a GPU; "tiny" is a smoke test

# Environment overrides (used by the repository's notebook test).
DATA_DIR = Path(_env("MB_DATA_DIR", "/content/malecns_v1_0"))
OUT_DIR = Path(_env("MB_OUT_DIR", "/content/meta_brain_m0_m1"))
OFFLINE = _env("MB_OFFLINE", "0") == "1"
MIN_NEURONS = _env("MB_MIN_NEURONS", MIN_NEURONS, int)
MIN_EDGES = _env("MB_MIN_EDGES", MIN_EDGES, int)
BENCH = _env("MB_BENCH", BENCH)
MICRO_STEPS = _env("MB_MICRO_STEPS", MICRO_STEPS, int)
DEVICE = torch.device(_env("MB_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
USE_DRIVE = USE_DRIVE and IN_COLAB and not OFFLINE
if DEVICE.type != "cuda" and BENCH == "full":
    print("No CUDA GPU: the M1 benchmark falls back to a tiny CPU grid and M1 is not evaluated.")
    BENCH = "tiny"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = DATA_DIR / "cache"

try:
    import psutil
    RAM_GIB = round(psutil.virtual_memory().total / 2**30, 1)
except ImportError:
    RAM_GIB = None
GPU_NAME = torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None
GPU_GIB = (round(torch.cuda.get_device_properties(DEVICE).total_memory / 2**30, 1)
           if DEVICE.type == "cuda" else None)

RESULTS = {
    "experiment": "Meta-Brain M0/M1 infrastructure",
    "source_digest": SOURCE_DIGEST,
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "device": str(DEVICE), "gpu": GPU_NAME, "gpu_gib": GPU_GIB, "ram_gib": RAM_GIB,
                    "colab": IN_COLAB},
    "config": {"statuses": list(STATUSES), "min_weight": MIN_WEIGHT, "micro_steps": MICRO_STEPS,
               "bench": BENCH, "min_neurons": MIN_NEURONS, "min_edges": MIN_EDGES},
}
print(json.dumps(RESULTS["environment"], indent=2))

In [ ]:
# Optional: keep the 1.2 GB download and the graph cache on Google Drive between sessions.
DRIVE_DATA = None
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DATA = Path("/content/drive/MyDrive/meta_brain/malecns_v1_0")
        DRIVE_DATA.mkdir(parents=True, exist_ok=True)
        CACHE_DIR = DRIVE_DATA.parent / "graph_cache"
        print("Drive cache:", DRIVE_DATA, "| graph cache:", CACHE_DIR)
    except Exception as exc:  # Drive is a convenience, never a requirement
        print("Drive not mounted:", type(exc).__name__, str(exc)[:120])
print("data:", DATA_DIR, "| graph cache:", CACHE_DIR, "| outputs:", OUT_DIR)

## 1. Dados oficiais do MaleCNS v1.0

São os três arquivos da *flat connectome* publicados em [male-cns.janelia.org/download](https://male-cns.janelia.org/download/). O de pesos tem ~152 milhões de linhas, uma por par de corpos conectados. A maioria envolve fragmentos não rastreados; com só os neurônios `Traced` sobram ~25,6 milhões de arestas.

In [ ]:
import requests

BASE = "https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome"
FILES = [
    "body-annotations-male-cns-v1.0-minconf-0.5.feather",
    "body-neurotransmitters-male-cns-v1.0.feather",
    "connectome-weights-male-cns-v1.0-minconf-0.5.feather",
]


def fetch(name):
    path = DATA_DIR / name
    if path.exists() and path.stat().st_size > 1000:
        return f"reusing {name} ({path.stat().st_size / 1e9:.2f} GB)"
    if DRIVE_DATA is not None and (DRIVE_DATA / name).exists():
        shutil.copy2(DRIVE_DATA / name, path)
        return f"copied {name} from Drive"
    tmp = path.with_suffix(".part")
    with requests.get(f"{BASE}/{name}", stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done, last = 0, 0
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 << 20):
                f.write(chunk)
                done += len(chunk)
                if done - last > 200e6:
                    print(f"  {name}: {done / 1e9:.2f}/{total / 1e9:.2f} GB", flush=True)
                    last = done
    tmp.replace(path)
    if DRIVE_DATA is not None:
        shutil.copy2(path, DRIVE_DATA / name)
    return f"downloaded {name} ({path.stat().st_size / 1e9:.2f} GB)"


if OFFLINE:
    print("offline: using files already in", DATA_DIR)
else:
    t0 = time.time()
    for name in FILES:
        print(fetch(name), flush=True)
    print(f"data ready in {time.time() - t0:.0f}s")
print(sorted(p.name for p in DATA_DIR.glob("*.feather")))

## 2. Biblioteca embutida

As quatro células abaixo são cópias de `meta_brain/*.py` cujo hash está em `SOURCE_DIGEST`, sem os imports relativos do pacote. Não há nada para configurar nelas.

In [ ]:
# --- embedded from meta_brain/graph.py ---
"""Vectorized MaleCNS connectome loader with an on-disk cache (milestone M0).

The MaleCNS v1.0 "flat connectome" release ships three files (the ones
``scripts/download_data.py`` fetches):

* ``body-annotations-male-cns-v1.0-minconf-0.5.feather``: one row per body
  (``bodyId``, ``status``, ``type``, ``superclass``, ...);
* ``body-neurotransmitters-male-cns-v1.0.feather``: per-body NT predictions
  (``body``, ``consensus_nt``, ``predicted_nt``);
* ``connectome-weights-male-cns-v1.0-minconf-0.5.feather``: ~152M weighted
  rows (``body_pre``, ``body_post``, ``weight``), one per connected body pair,
  including the many untraced fragments. Keeping ``Traced`` bodies leaves
  ~25.6M edges. Repeated pairs (none in v1.0) would be summed.

The legacy layout (``*meta*.feather`` + ``*simple*edgelist*.feather``) is still
accepted.

The weights table is streamed one Arrow record batch at a time, body ids are
remapped with ``searchsorted``, and duplicate pairs are merged incrementally so
peak memory stays bounded by the number of unique pairs rather than the number
of rows. Every row is accounted for in :class:`LoadReport`.

The resulting :class:`ConnectomeArrays` stores edges sorted by ``(post, pre)``,
which is CSR row order for :mod:`meta_brain.sparse_ops`.
"""


import hashlib
import json
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

LOADER_VERSION = 2

# Candidate column names, in priority order. The first one present is used.
META_COLUMNS: Dict[str, Tuple[str, ...]] = {
    "body_id": ("bodyId", "body_id", "body", "root_id"),
    "status": ("status",),
    "type": ("type", "cell_type"),
    "superclass": ("superclass", "super_class"),
    "class": ("class", "cell_class"),
    "nt": ("predictedNt", "consensusNt", "nt_type", "predicted_nt"),
    "nt_confidence": ("predictedNtConfidence", "predictedNtProb", "nt_type_score"),
    "side": ("somaSide", "rootSide", "side"),
}
NT_FILE_COLUMNS: Dict[str, Tuple[str, ...]] = {
    "body_id": ("body", "bodyId", "body_id"),
    "consensus": ("consensus_nt", "consensusNt"),
    "predicted": ("predicted_nt", "predictedNt"),
    "confidence": ("predicted_nt_prob", "predicted_nt_confidence", "predictedNtProb",
                   "predictedNtConfidence"),
}
EDGE_COLUMNS: Dict[str, Tuple[str, ...]] = {
    "pre": ("body_pre", "bodyId_pre", "pre", "pre_root_id"),
    "post": ("body_post", "bodyId_post", "post", "post_root_id"),
    "weight": ("weight", "syn_count", "count"),
}

FLAT_PATTERNS = {
    "annotations": "body-annotations-*.feather",
    "neurotransmitters": "body-neurotransmitters-*.feather",
    "edges": "connectome-weights-*.feather",
}
LEGACY_PATTERNS = {"annotations": "*meta*.feather", "edges": "*simple*edgelist*.feather"}

DEFAULT_STATUSES: Tuple[str, ...] = ("Traced",)
DEFAULT_MERGE_EVERY = 30_000_000  # rows accumulated before an incremental merge

# Neurotransmitter codes stored per neuron. Signs are a separate, overridable
# mapping so downstream layers (e.g. dopamine as a teaching signal) can decide
# how to treat modulatory transmitters.
NT_CODES: Dict[str, int] = {
    "unknown": 0,
    "acetylcholine": 1,
    "gaba": 2,
    "glutamate": 3,
    "histamine": 4,
    "dopamine": 5,
    "serotonin": 6,
    "octopamine": 7,
    "tyramine": 8,
}
NT_ALIASES: Dict[str, str] = {
    "ach": "acetylcholine",
    "glu": "glutamate",
    "da": "dopamine",
    "5ht": "serotonin",
    "ser": "serotonin",
    "oa": "octopamine",
    "ta": "tyramine",
    "his": "histamine",
}
# Fast-transmission sign by NT code. Glutamate is inhibitory at most fly
# central synapses (GluCl) and histamine is the inhibitory photoreceptor
# transmitter. Modulatory and unknown transmitters default to +1; layer 2
# handles dopamine separately through ``nt_code``.
DEFAULT_NT_SIGN: Dict[str, float] = {
    "unknown": 1.0,
    "acetylcholine": 1.0,
    "gaba": -1.0,
    "glutamate": -1.0,
    "histamine": -1.0,
    "dopamine": 1.0,
    "serotonin": 1.0,
    "octopamine": 1.0,
    "tyramine": 1.0,
}

SENSORY_SUPERCLASSES = ("ol_sensory", "cb_sensory", "vnc_sensory", "sensory_ascending", "olsn")
MOTOR_SUPERCLASSES = ("descending_neuron", "vnc_motor", "cb_motor", "dn", "motor")
DESCENDING_SUPERCLASSES = ("descending_neuron", "dn")


@dataclass
class DataFiles:
    annotations: Path
    edges: Path
    neurotransmitters: Optional[Path] = None
    layout: str = "flat"

    def all(self) -> List[Path]:
        return [p for p in (self.annotations, self.neurotransmitters, self.edges) if p is not None]


@dataclass
class LoadReport:
    """Every body and edge row that was read, dropped or merged, so counts can be audited."""

    layout: str = ""
    files: Dict[str, str] = field(default_factory=dict)
    columns_used: Dict[str, str] = field(default_factory=dict)
    statuses: List[str] = field(default_factory=list)
    bodies_annotated: int = 0
    dropped_by_status: int = 0
    n_neurons: int = 0
    nt_source: str = ""
    edge_rows_read: int = 0
    dropped_unknown_pre: int = 0
    dropped_unknown_post: int = 0
    dropped_self_loops: int = 0
    duplicate_rows_merged: int = 0
    dropped_below_min_weight: int = 0
    min_weight: float = 1.0
    edges_retained: int = 0
    synapses_retained: int = 0
    n_cell_types: int = 0
    n_untyped_neurons: int = 0
    n_type_pairs: int = 0
    nt_counts: Dict[str, int] = field(default_factory=dict)
    build_seconds: float = 0.0
    load_seconds: float = 0.0
    from_cache: bool = False

    def accounted_rows(self) -> int:
        return (self.edges_retained + self.dropped_unknown_pre + self.dropped_unknown_post
                + self.dropped_self_loops + self.duplicate_rows_merged + self.dropped_below_min_weight)

    def summary(self) -> str:
        status = f" status in {self.statuses}" if self.statuses else ""
        lines = [
            f"layout={self.layout}  bodies annotated={self.bodies_annotated:,}"
            f"  dropped by{status or ' status'}={self.dropped_by_status:,}",
            f"neurons={self.n_neurons:,}  NT source={self.nt_source}",
            f"edge rows read={self.edge_rows_read:,}",
            f"  dropped unknown pre={self.dropped_unknown_pre:,} post={self.dropped_unknown_post:,}",
            f"  dropped self-loops={self.dropped_self_loops:,}",
            f"  duplicate rows merged={self.duplicate_rows_merged:,}",
            f"  pairs below min weight {self.min_weight:g}={self.dropped_below_min_weight:,}",
            f"edges retained={self.edges_retained:,} (synapses={self.synapses_retained:,})",
            f"cell types={self.n_cell_types:,} (untyped neurons={self.n_untyped_neurons:,}), "
            f"type pairs={self.n_type_pairs:,}",
            f"build time={self.build_seconds:.1f}s"
            + (f", cache load time={self.load_seconds:.1f}s" if self.from_cache else ""),
        ]
        return "\n".join(lines)


@dataclass
class ConnectomeArrays:
    """Canonical sparse connectome. Edges are sorted by ``(post, pre)``."""

    body_id: np.ndarray  # (N,) int64
    pre: np.ndarray  # (E,) int32, neuron index
    post: np.ndarray  # (E,) int32, neuron index
    count: np.ndarray  # (E,) float32, synapse count summed over rows
    nt_code: np.ndarray  # (N,) int8, see NT_CODES
    nt_confidence: np.ndarray  # (N,) float32, NaN when unavailable
    cell_type: np.ndarray  # (N,) int32 index into type_names
    type_names: List[str]
    superclass: np.ndarray  # (N,) int16 index into superclass_names
    superclass_names: List[str]
    side: np.ndarray  # (N,) int8: 0 unknown, 1 left, 2 right, 3 center
    report: LoadReport = field(default_factory=LoadReport)

    @property
    def n(self) -> int:
        return int(len(self.body_id))

    @property
    def e(self) -> int:
        return int(len(self.pre))

    @property
    def n_types(self) -> int:
        return len(self.type_names)

    def sign(self, nt_sign: Optional[Dict[str, float]] = None) -> np.ndarray:
        """Per-neuron fast-transmission sign from the NT prediction."""
        table = dict(DEFAULT_NT_SIGN)
        if nt_sign:
            table.update(nt_sign)
        lut = np.ones(len(NT_CODES), dtype=np.float32)
        for name, code in NT_CODES.items():
            lut[code] = table.get(name, 1.0)
        return lut[self.nt_code]

    def superclass_mask(self, names: Sequence[str]) -> np.ndarray:
        wanted = {s.lower() for s in names}
        codes = [i for i, s in enumerate(self.superclass_names) if s in wanted]
        return np.isin(self.superclass, codes)

    def sensory_indices(self) -> np.ndarray:
        return np.flatnonzero(self.superclass_mask(SENSORY_SUPERCLASSES))

    def motor_indices(self) -> np.ndarray:
        return np.flatnonzero(self.superclass_mask(MOTOR_SUPERCLASSES))

    def descending_indices(self) -> np.ndarray:
        return np.flatnonzero(self.superclass_mask(DESCENDING_SUPERCLASSES))

    def type_pair_index(self) -> Tuple[np.ndarray, np.ndarray]:
        """Map every edge to a connected ``(pre_type, post_type)`` pair.

        Returns ``(edge_pair, pair_types)`` where ``edge_pair`` is ``(E,)``
        int32 and ``pair_types`` is ``(P, 2)`` int32 with columns
        ``[pre_type, post_type]``.
        """
        return type_pair_index(self.pre, self.post, self.cell_type, self.n_types)

    def csr_rowptr(self) -> np.ndarray:
        """CSR row pointer with rows = postsynaptic neuron."""
        counts = np.bincount(self.post, minlength=self.n)
        rowptr = np.zeros(self.n + 1, dtype=np.int64)
        np.cumsum(counts, out=rowptr[1:])
        return rowptr

    def replace_edges(self, pre: np.ndarray, post: np.ndarray, count: np.ndarray) -> "ConnectomeArrays":
        """Copy with a different edge set (used by the shuffle controls)."""
        pre, post, count = sort_edges(pre, post, count)
        out = ConnectomeArrays(**{**self.__dict__, "pre": pre, "post": post, "count": count})
        out.report = LoadReport(**{**asdict(self.report), "edges_retained": len(pre),
                                   "synapses_retained": int(count.sum())})
        return out


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

def _pick(columns: Sequence[str], candidates: Sequence[str], required: bool, what: str) -> Optional[str]:
    for c in candidates:
        if c in columns:
            return c
    if required:
        raise ValueError(f"{what}: none of {list(candidates)} found in columns {list(columns)}")
    return None


def _nt_names(values) -> pd.Series:
    names = pd.Series(values).fillna("").astype(str).str.strip().str.lower()
    return names.map(lambda s: NT_ALIASES.get(s, s))


def _nt_codes(names: pd.Series) -> np.ndarray:
    return names.map(lambda s: NT_CODES.get(s, 0)).to_numpy(dtype=np.int8)


def _normalize_side(values: Optional[pd.Series], n: int) -> np.ndarray:
    if values is None:
        return np.zeros(n, dtype=np.int8)
    s = values.fillna("").astype(str).str.strip().str.lower().str[:1]
    return s.map({"l": 1, "r": 2, "c": 3, "m": 3}).fillna(0).to_numpy(dtype=np.int8)


def cell_type_keys(meta: pd.DataFrame, type_col: Optional[str], superclass_col: Optional[str],
                   class_col: Optional[str]) -> Tuple[np.ndarray, int]:
    """Type key per neuron; untyped neurons fall back to ``untyped:<superclass>:<class>``."""
    n = len(meta)

    def col(name: Optional[str]) -> pd.Series:
        if name is None:
            return pd.Series([""] * n, index=meta.index)
        return meta[name].fillna("").astype(str).str.strip()

    typ, sup, cls = col(type_col), col(superclass_col), col(class_col)
    untyped = (typ == "") | typ.str.lower().isin({"nan", "none", "unknown"})
    keys = typ.where(~untyped, "untyped:" + sup.str.lower() + ":" + cls.str.lower())
    return keys.to_numpy(dtype=object), int(untyped.sum())


def remap_ids(sorted_ids: np.ndarray, order: np.ndarray, raw: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Vectorized bodyId -> row index. Returns ``(index, valid_mask)``."""
    if len(sorted_ids) == 0:
        return np.zeros(len(raw), np.int32), np.zeros(len(raw), bool)
    pos = np.searchsorted(sorted_ids, raw)
    pos = np.minimum(pos, len(sorted_ids) - 1)
    valid = sorted_ids[pos] == raw
    return order[pos].astype(np.int32), valid


def sort_edges(pre: np.ndarray, post: np.ndarray, count: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    order = np.lexsort((pre, post))
    return (pre[order].astype(np.int32, copy=False), post[order].astype(np.int32, copy=False),
            count[order].astype(np.float32, copy=False))


def merge_duplicates(pre: np.ndarray, post: np.ndarray, count: np.ndarray, n: int
                     ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int]:
    """Sum the counts of repeated ``(pre, post)`` rows. Returns the merged edges."""
    key = post.astype(np.int64) * n + pre.astype(np.int64)
    uniq, inverse = np.unique(key, return_inverse=True)
    merged = len(key) - len(uniq)
    if merged == 0:
        return (*sort_edges(pre, post, count), 0)
    summed = np.bincount(inverse, weights=count.astype(np.float64), minlength=len(uniq))
    return ((uniq % n).astype(np.int32), (uniq // n).astype(np.int32),
            summed.astype(np.float32), int(merged))


def _merge_keys(keys: List[np.ndarray], weights: List[np.ndarray]) -> Tuple[np.ndarray, np.ndarray]:
    """Merge ``post * n + pre`` keys, summing weights; output is sorted and unique."""
    k = np.concatenate(keys)
    w = np.concatenate(weights)
    uniq, inverse = np.unique(k, return_inverse=True)
    return uniq, np.bincount(inverse, weights=w, minlength=len(uniq))


def type_pair_index(pre: np.ndarray, post: np.ndarray, cell_type: np.ndarray, n_types: int
                    ) -> Tuple[np.ndarray, np.ndarray]:
    key = cell_type[pre].astype(np.int64) * n_types + cell_type[post].astype(np.int64)
    uniq, inverse = np.unique(key, return_inverse=True)
    pair_types = np.stack([uniq // n_types, uniq % n_types], axis=1).astype(np.int32)
    return inverse.astype(np.int32), pair_types


def _feather_columns(path: Path) -> List[str]:
    import pyarrow as pa
    import pyarrow.ipc as ipc

    try:
        return list(ipc.open_file(pa.memory_map(str(path), "r")).schema.names)
    except pa.ArrowInvalid:
        return list(pd.read_feather(path).columns)


def _iter_edge_batches(path: Path, columns: List[str]) -> Iterator[Dict[str, np.ndarray]]:
    """Yield the edge table one Arrow record batch at a time (bounded memory)."""
    import pyarrow as pa
    import pyarrow.ipc as ipc

    try:
        reader = ipc.open_file(pa.memory_map(str(path), "r"))
    except pa.ArrowInvalid:
        # Feather v1 or a non-IPC file: fall back to a single full read.
        df = pd.read_feather(path, columns=columns)
        yield {c: df[c].to_numpy() for c in columns}
        return
    for i in range(reader.num_record_batches):
        batch = reader.get_batch(i)
        yield {c: batch.column(c).to_numpy(zero_copy_only=False) for c in columns}


def find_data_files(data_dir: str | Path) -> DataFiles:
    """Locate the flat-connectome release, falling back to the legacy layout."""
    data_dir = Path(data_dir)

    def first(pattern: str) -> Optional[Path]:
        hits = sorted(data_dir.glob(pattern))
        return hits[0] if hits else None

    ann, edges = first(FLAT_PATTERNS["annotations"]), first(FLAT_PATTERNS["edges"])
    if ann and edges:
        return DataFiles(ann, edges, first(FLAT_PATTERNS["neurotransmitters"]), "flat")
    ann, edges = first(LEGACY_PATTERNS["annotations"]), first(LEGACY_PATTERNS["edges"])
    if ann and edges:
        return DataFiles(ann, edges, None, "legacy")
    raise FileNotFoundError(
        f"No connectome in {data_dir}. Expected {FLAT_PATTERNS['annotations']} + "
        f"{FLAT_PATTERNS['edges']} (+ {FLAT_PATTERNS['neurotransmitters']}); "
        "run scripts/download_data.py.")


def _join_neurotransmitters(nt_path: Path, body_id: np.ndarray, report: LoadReport
                            ) -> Tuple[np.ndarray, np.ndarray]:
    """Consensus NT where available and not 'unclear', else the per-body prediction."""
    cols = _feather_columns(nt_path)
    used = {k: _pick(cols, cands, k in ("body_id",), f"neurotransmitters {k}")
            for k, cands in NT_FILE_COLUMNS.items()}
    if used["consensus"] is None and used["predicted"] is None:
        raise ValueError(f"neurotransmitters: no consensus/predicted column in {cols}")
    report.columns_used.update({f"nt.{k}": v for k, v in used.items() if v})
    nt_df = pd.read_feather(nt_path, columns=[v for v in used.values() if v])
    nt_df = nt_df.drop_duplicates(used["body_id"]).set_index(used["body_id"])

    def column(key: str) -> pd.Series:
        if used[key] is None:
            return pd.Series([""] * len(body_id))
        return _nt_names(nt_df[used[key]].reindex(body_id).to_numpy())

    cons, pred = column("consensus"), column("predicted")
    usable = (cons != "") & (cons != "unclear")
    names = cons.where(usable, pred)
    report.nt_source = "consensus_nt, else predicted_nt" if used["consensus"] else "predicted_nt"
    conf = (pd.to_numeric(nt_df[used["confidence"]].reindex(body_id), errors="coerce")
            .to_numpy(dtype=np.float32) if used["confidence"]
            else np.full(len(body_id), np.nan, dtype=np.float32))
    return _nt_codes(names), conf


# --------------------------------------------------------------------------- #
# Loading and caching
# --------------------------------------------------------------------------- #

def build_connectome(files: DataFiles, min_weight: float = 1.0, drop_self_loops: bool = True,
                     statuses: Optional[Sequence[str]] = DEFAULT_STATUSES, verbose: bool = True,
                     merge_every: int = DEFAULT_MERGE_EVERY) -> ConnectomeArrays:
    """Build :class:`ConnectomeArrays` from MaleCNS feather files.

    ``statuses`` keeps only bodies whose ``status`` is listed (``None`` keeps
    all). ``min_weight`` is applied to the *merged* pair weight, i.e. the
    total synapse count between two neurons.
    """
    t0 = time.perf_counter()
    report = LoadReport(layout=files.layout, min_weight=float(min_weight),
                        files={k: str(v) for k, v in asdict(files).items() if isinstance(v, Path)})

    meta = pd.read_feather(files.annotations)
    cols = list(meta.columns)
    used = {k: _pick(cols, cands, k == "body_id", f"annotations {k}") for k, cands in META_COLUMNS.items()}
    report.columns_used.update({f"meta.{k}": v for k, v in used.items() if v})
    report.bodies_annotated = len(meta)
    if statuses and used["status"]:
        keep = meta[used["status"]].astype(str).isin(set(statuses))
        report.dropped_by_status = int((~keep).sum())
        report.statuses = list(statuses)
        meta = meta[keep.to_numpy()].reset_index(drop=True)

    body_id = meta[used["body_id"]].astype(np.int64).to_numpy()
    if len(np.unique(body_id)) != len(body_id):
        raise ValueError("Neuron annotations contain duplicate body ids.")
    n = len(body_id)
    report.n_neurons = n

    keys, n_untyped = cell_type_keys(meta, used["type"], used["superclass"], used["class"])
    type_names, cell_type = np.unique(keys.astype(str), return_inverse=True)
    report.n_cell_types, report.n_untyped_neurons = len(type_names), n_untyped

    sup = (meta[used["superclass"]].fillna("").astype(str).str.strip().str.lower()
           if used["superclass"] else pd.Series([""] * n))
    superclass_names, superclass = np.unique(sup.to_numpy(dtype=str), return_inverse=True)

    if files.neurotransmitters is not None:
        nt_code, nt_conf = _join_neurotransmitters(files.neurotransmitters, body_id, report)
    else:
        nt_code = (_nt_codes(_nt_names(meta[used["nt"]].to_numpy())) if used["nt"]
                   else np.zeros(n, dtype=np.int8))
        nt_conf = (pd.to_numeric(meta[used["nt_confidence"]], errors="coerce").to_numpy(dtype=np.float32)
                   if used["nt_confidence"] else np.full(n, np.nan, dtype=np.float32))
        report.nt_source = f"annotations.{used['nt']}" if used["nt"] else "none"
    inv_nt = {v: k for k, v in NT_CODES.items()}
    report.nt_counts = {inv_nt[int(c)]: int(k) for c, k in zip(*np.unique(nt_code, return_counts=True))}
    side = _normalize_side(meta[used["side"]] if used["side"] else None, n)
    del meta

    order = np.argsort(body_id, kind="stable")
    sorted_ids = body_id[order]

    edge_cols = _feather_columns(files.edges)
    ecol = {k: _pick(edge_cols, cands, k != "weight", f"edges {k}") for k, cands in EDGE_COLUMNS.items()}
    report.columns_used.update({f"edge.{k}": v for k, v in ecol.items() if v})

    uniq_keys = np.zeros(0, np.int64)
    uniq_w = np.zeros(0, np.float64)
    pend_k: List[np.ndarray] = []
    pend_w: List[np.ndarray] = []
    pending = kept_rows = 0
    for batch in _iter_edge_batches(files.edges, [c for c in ecol.values() if c]):
        raw_pre = np.asarray(batch[ecol["pre"]], dtype=np.int64)
        raw_post = np.asarray(batch[ecol["post"]], dtype=np.int64)
        w = (np.asarray(batch[ecol["weight"]], dtype=np.float64) if ecol["weight"]
             else np.ones(len(raw_pre), np.float64))
        report.edge_rows_read += len(w)

        pre_i, ok_pre = remap_ids(sorted_ids, order, raw_pre)
        post_i, ok_post = remap_ids(sorted_ids, order, raw_post)
        report.dropped_unknown_pre += int((~ok_pre).sum())
        report.dropped_unknown_post += int((ok_pre & ~ok_post).sum())
        keep = ok_pre & ok_post
        if drop_self_loops:
            loops = keep & (pre_i == post_i)
            report.dropped_self_loops += int(loops.sum())
            keep &= ~loops

        pend_k.append(post_i[keep].astype(np.int64) * n + pre_i[keep])
        pend_w.append(w[keep])
        pending += int(keep.sum())
        kept_rows += int(keep.sum())
        if pending >= merge_every:
            uniq_keys, uniq_w = _merge_keys([uniq_keys, *pend_k], [uniq_w, *pend_w])
            pend_k, pend_w, pending = [], [], 0
            if verbose:
                print(f"  rows read {report.edge_rows_read:,} -> unique pairs {len(uniq_keys):,}",
                      flush=True)
    uniq_keys, uniq_w = _merge_keys([uniq_keys, *pend_k], [uniq_w, *pend_w])
    del pend_k, pend_w
    report.duplicate_rows_merged = kept_rows - len(uniq_keys)

    strong = uniq_w >= min_weight
    report.dropped_below_min_weight = int((~strong).sum())
    uniq_keys, uniq_w = uniq_keys[strong], uniq_w[strong]
    pre = (uniq_keys % n).astype(np.int32)
    post = (uniq_keys // n).astype(np.int32)
    count = uniq_w.astype(np.float32)
    del uniq_keys
    report.edges_retained = len(pre)
    report.synapses_retained = int(uniq_w.sum())

    graph = ConnectomeArrays(
        body_id=body_id, pre=pre, post=post, count=count, nt_code=nt_code,
        nt_confidence=nt_conf, cell_type=cell_type.astype(np.int32),
        type_names=[str(t) for t in type_names], superclass=superclass.astype(np.int16),
        superclass_names=[str(s) for s in superclass_names], side=side, report=report,
    )
    report.n_type_pairs = int(len(graph.type_pair_index()[1]))
    report.load_seconds = report.build_seconds = time.perf_counter() - t0
    if report.accounted_rows() != report.edge_rows_read:
        raise AssertionError(f"Edge accounting mismatch: {report.accounted_rows()} != {report.edge_rows_read}")
    if verbose:
        print(report.summary())
    return graph


_ARRAY_FIELDS = ("body_id", "pre", "post", "count", "nt_code", "nt_confidence",
                 "cell_type", "superclass", "side")


def _cache_key(files: DataFiles, min_weight: float, drop_self_loops: bool,
               statuses: Optional[Sequence[str]]) -> str:
    parts = [str(LOADER_VERSION), str(min_weight), str(drop_self_loops),
             ",".join(sorted(statuses)) if statuses else "*"]
    for p in files.all():
        st = p.stat()
        parts += [p.name, str(st.st_size), str(int(st.st_mtime))]
    return hashlib.sha1("|".join(parts).encode()).hexdigest()[:16]


def save_cache(graph: ConnectomeArrays, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez(tmp, **{k: getattr(graph, k) for k in _ARRAY_FIELDS})
    tmp.replace(path)
    side = {"type_names": graph.type_names, "superclass_names": graph.superclass_names,
            "report": asdict(graph.report)}
    path.with_suffix(".json").write_text(json.dumps(side), encoding="utf-8")


def load_cache(path: Path) -> ConnectomeArrays:
    t0 = time.perf_counter()
    side = json.loads(path.with_suffix(".json").read_text(encoding="utf-8"))
    with np.load(path) as data:
        arrays = {k: data[k] for k in _ARRAY_FIELDS}
    report = LoadReport(**side["report"])
    report.from_cache = True
    report.load_seconds = time.perf_counter() - t0
    return ConnectomeArrays(**arrays, type_names=side["type_names"],
                            superclass_names=side["superclass_names"], report=report)


def load_connectome(data_dir: str | Path = "data/connectome", cache_dir: str | Path | None = None,
                    min_weight: float = 1.0, drop_self_loops: bool = True,
                    statuses: Optional[Sequence[str]] = DEFAULT_STATUSES, use_cache: bool = True,
                    verbose: bool = True) -> ConnectomeArrays:
    """Load the connectome from ``data_dir``, building and caching it on first use.

    The cache is keyed on the source files' names, sizes and mtimes plus the
    loader options, so a new data release or option change rebuilds it.
    """
    data_dir = Path(data_dir)
    files = find_data_files(data_dir)
    cache_dir = Path(cache_dir) if cache_dir is not None else data_dir / "cache"
    cache = cache_dir / f"connectome_{_cache_key(files, min_weight, drop_self_loops, statuses)}.npz"
    if use_cache and cache.exists() and cache.with_suffix(".json").exists():
        graph = load_cache(cache)
        if verbose:
            print(f"Loaded cached connectome {cache.name}\n{graph.report.summary()}")
        return graph
    graph = build_connectome(files, min_weight, drop_self_loops, statuses, verbose)
    if use_cache:
        save_cache(graph, cache)
    return graph


In [ ]:
# --- embedded from meta_brain/shuffle.py ---
"""Null-model graphs for the connectome controls (milestone M0).

Each generator returns new ``(pre, post, count)`` arrays with the same number
of neurons and, as far as possible, the same number of edges as the input.
Signs live on neurons (via the presynaptic NT), so they are preserved
automatically by any generator that keeps the presynaptic endpoint of an edge.

* :func:`degree_preserving_shuffle` permutes postsynaptic endpoints across all
  edges: every neuron keeps its exact in- and out-degree, and its out-strength.
* :func:`type_preserving_shuffle` rewires only within a ``(pre_type,
  post_type)`` block, by repeated edge swaps: the type-to-type connectivity
  table and each neuron's per-block degrees are kept; only individual wiring
  (e.g. retinotopy) is destroyed. Every swap keeps the graph simple.
* :func:`random_sparse_graph` keeps N, E and the weight multiset but nothing
  else.

The degree-preserving shuffle and the random graph create some self-loops and
duplicate edges. They are repaired with degree-preserving swaps; whatever
cannot be repaired within ``max_rounds`` is dropped (self-loops) or merged
(duplicates) and reported in :class:`ShuffleReport`.
"""


from dataclasses import dataclass
from typing import Optional, Tuple

import numpy as np



@dataclass
class ShuffleReport:
    method: str
    edges_in: int
    edges_out: int
    repair_rounds: int
    residual_self_loops_dropped: int
    residual_duplicates_merged: int
    swap_sweeps: int = 0
    swaps_accepted: int = 0

    @property
    def exact(self) -> bool:
        return self.residual_self_loops_dropped == 0 and self.residual_duplicates_merged == 0


Edges = Tuple[np.ndarray, np.ndarray, np.ndarray]


def _bad_edges(pre: np.ndarray, post: np.ndarray, n: int) -> np.ndarray:
    """Mask of self-loops and every-but-the-first copy of a duplicate edge."""
    key = post.astype(np.int64) * n + pre.astype(np.int64)
    _, first = np.unique(key, return_index=True)
    dup = np.ones(len(key), dtype=bool)
    dup[first] = False
    return dup | (pre == post)


def _repair(pre: np.ndarray, post: np.ndarray, group: np.ndarray, n: int,
            rng: np.random.Generator, max_rounds: int) -> Tuple[np.ndarray, int]:
    """Swap the postsynaptic endpoint of bad edges with random edges of the same group.

    A swap between two edges of the same group preserves every per-group in-
    and out-degree, so the repair never breaks the shuffle's invariants. A swap
    is only accepted when both resulting edges are new and not self-loops, so
    the number of bad edges never increases.
    """
    pre64 = pre.astype(np.int64)
    post = post.astype(np.int64)
    order = np.argsort(group, kind="stable")
    groups, start, size = np.unique(group[order], return_index=True, return_counts=True)
    gpos = np.searchsorted(groups, group)
    rounds = stalled = 0
    for rounds in range(1, max_rounds + 1):
        bad = np.flatnonzero(_bad_edges(pre64, post, n))
        if len(bad) == 0:
            return post.astype(np.int32), rounds - 1
        existing = np.sort(post * n + pre64)

        def present(k: np.ndarray) -> np.ndarray:
            i = np.minimum(np.searchsorted(existing, k), len(existing) - 1)
            return existing[i] == k

        # Hub-to-hub duplicates rarely find a valid partner at random, so try
        # several candidates per bad edge and keep the first valid one.
        c = int(np.clip((1 << 20) // len(bad), 4, 512))
        bb = np.repeat(bad, c)
        gg = gpos[bb]
        pp = order[start[gg] + (rng.random(len(bb)) * size[gg]).astype(np.int64)]
        valid = ((bb != pp) & (pre64[bb] != post[pp]) & (pre64[pp] != post[bb])
                 & ~present(post[pp] * n + pre64[bb]) & ~present(post[bb] * n + pre64[pp]))
        valid = valid.reshape(len(bad), c)
        has = valid.any(axis=1)
        b = bad[has]
        p = pp.reshape(len(bad), c)[has, valid[has].argmax(axis=1)]

        # Every edge may take part in at most one swap per round, and two
        # accepted swaps must not create the same edge.
        _, first = np.unique(np.concatenate([b, p]), return_index=True)
        once = np.zeros(2 * len(b), dtype=bool)
        once[first] = True
        ok = once[: len(b)] & once[len(b):]
        b, p = b[ok], p[ok]
        new_b, new_p = post[p] * n + pre64[b], post[b] * n + pre64[p]
        _, inv, counts = np.unique(np.concatenate([new_b, new_p]), return_inverse=True,
                                   return_counts=True)
        clash = counts[inv] > 1
        ok = ~(clash[: len(b)] | clash[len(b):])
        b, p = b[ok], p[ok]
        post[b], post[p] = post[p], post[b].copy()
        # No valid partner among hundreds of candidates for several rounds: the
        # leftovers are (almost surely) infeasible within their block.
        stalled = stalled + 1 if len(b) == 0 else 0
        if stalled >= 5:
            break
    return post.astype(np.int32), rounds


def _finish(method: str, pre: np.ndarray, post: np.ndarray, count: np.ndarray, n: int,
            rounds: int) -> Tuple[Edges, ShuffleReport]:
    loops = pre == post
    pre, post, count = pre[~loops], post[~loops], count[~loops]
    edges_before = len(pre)
    pre, post, count, merged = merge_duplicates(pre, post, count, n)
    report = ShuffleReport(method, edges_before + int(loops.sum()), len(pre), rounds,
                           int(loops.sum()), merged)
    return (pre, post, count), report


def degree_preserving_shuffle(pre: np.ndarray, post: np.ndarray, count: np.ndarray, n: int,
                              seed: Optional[int] = None, max_rounds: int = 100
                              ) -> Tuple[Edges, ShuffleReport]:
    rng = np.random.default_rng(seed)
    group = np.zeros(len(pre), dtype=np.int64)
    new_post = rng.permutation(post)
    new_post, rounds = _repair(pre, new_post, group, n, rng, max_rounds)
    return _finish("degree_preserving", pre.copy(), new_post, count.copy(), n, rounds)


def _swap_sweeps(pre: np.ndarray, post: np.ndarray, group: np.ndarray, n: int,
                 rng: np.random.Generator, sweeps: int) -> Tuple[np.ndarray, int]:
    """Edge-swap rewiring: exchange postsynaptic endpoints of random edge pairs in the same group.

    Each sweep pairs every edge with a random partner of its group and accepts
    the swaps whose two new edges are neither self-loops nor already present
    (and do not collide with another accepted swap). The graph stays simple and
    every per-group in- and out-degree is preserved after every sweep, so no
    repair is needed.
    """
    pre64 = pre.astype(np.int64)
    post = post.astype(np.int64)
    accepted = 0
    if len(pre) < 2:
        return post.astype(np.int32), 0
    # Group id in the high bits, a fresh random number in the low bits: one argsort
    # gives a random order within every group.
    g_hi = np.unique(group, return_inverse=True)[1].astype(np.int64) << 31
    for _ in range(sweeps):
        order = np.argsort(g_hi | rng.integers(0, 1 << 31, len(pre)), kind="stable")
        a, b = order[0:-1:2], order[1::2]
        same = group[a] == group[b]
        a, b = a[same], b[same]

        existing = np.sort(post * n + pre64)

        def present(k: np.ndarray) -> np.ndarray:
            i = np.minimum(np.searchsorted(existing, k), len(existing) - 1)
            return existing[i] == k

        new_a, new_b = post[b] * n + pre64[a], post[a] * n + pre64[b]
        ok = ((post[a] != post[b]) & (pre64[a] != post[b]) & (pre64[b] != post[a])
              & ~present(new_a) & ~present(new_b))
        a, b, new_a, new_b = a[ok], b[ok], new_a[ok], new_b[ok]
        _, inv, counts = np.unique(np.concatenate([new_a, new_b]), return_inverse=True, return_counts=True)
        clash = counts[inv] > 1
        keep = ~(clash[: len(a)] | clash[len(a):])
        a, b = a[keep], b[keep]
        post[a], post[b] = post[b], post[a].copy()
        accepted += len(a)
    return post.astype(np.int32), accepted


def type_preserving_shuffle(pre: np.ndarray, post: np.ndarray, count: np.ndarray, n: int,
                            cell_type: np.ndarray, seed: Optional[int] = None,
                            sweeps: int = 10) -> Tuple[Edges, ShuffleReport]:
    """Rewire within ``(pre_type, post_type)`` blocks by repeated edge swaps.

    Permute-then-repair does not scale here: the real connectome has millions
    of small, dense type-pair blocks where a random permutation creates many
    duplicates that no swap can fix. Swapping keeps the graph valid at every
    step instead. Dense blocks simply accept few swaps, which shows up as a
    higher ``overlap with the original`` rather than as dropped edges.
    """
    rng = np.random.default_rng(seed)
    n_types = int(cell_type.max()) + 1 if len(cell_type) else 1
    group = cell_type[pre].astype(np.int64) * n_types + cell_type[post].astype(np.int64)
    new_post, accepted = _swap_sweeps(pre, post, group, n, rng, sweeps)
    edges, report = _finish("type_preserving", pre.copy(), new_post, count.copy(), n, 0)
    report.swap_sweeps, report.swaps_accepted = sweeps, accepted
    return edges, report


def random_sparse_graph(n: int, e: int, count: np.ndarray, seed: Optional[int] = None,
                        max_rounds: int = 100) -> Tuple[Edges, ShuffleReport]:
    """Uniform random directed graph with ``e`` edges and a permuted weight multiset."""
    rng = np.random.default_rng(seed)
    pre = rng.integers(0, n, e, dtype=np.int64)
    post = rng.integers(0, n, e, dtype=np.int64)
    rounds = 0
    for rounds in range(1, max_rounds + 1):
        bad = np.flatnonzero(_bad_edges(pre, post, n))
        if len(bad) == 0:
            rounds -= 1
            break
        pre[bad] = rng.integers(0, n, len(bad))
        post[bad] = rng.integers(0, n, len(bad))
    w = rng.permutation(count).astype(np.float32)
    return _finish("random_sparse", pre.astype(np.int32), post.astype(np.int32), w, n, rounds)


def in_out_degrees(pre: np.ndarray, post: np.ndarray, n: int) -> Tuple[np.ndarray, np.ndarray]:
    return np.bincount(post, minlength=n), np.bincount(pre, minlength=n)


__all__ = [
    "ShuffleReport",
    "degree_preserving_shuffle",
    "type_preserving_shuffle",
    "random_sparse_graph",
    "in_out_degrees",
    "sort_edges",
]


In [ ]:
# --- embedded from meta_brain/sparse_ops.py ---
"""Memory-lean sparse connectome propagation with a custom autograd (milestone M1).

``y = A(values) @ x`` where ``A`` is the ``N x N`` connectome with rows =
postsynaptic and columns = presynaptic neuron, and ``values`` holds one weight
per edge.

Plain autograd through a gather/scatter implementation saves an ``E``-sized
tensor (``x[pre] * w``) at every call. Unrolled over ``K`` micro-steps x ``T``
frames that is what makes BPTT through the full brain impossible. This
operator saves only ``x`` (``N x B``) per call and recomputes edge products in
the backward pass, in bounded chunks:

* ``grad_x      = A^T @ grad_y``                      (sparse CSR matmul)
* ``grad_values[e] = sum_b grad_y[post_e, b] * x[pre_e, b]``   (SDDMM via
  ``torch.sparse.sampled_addmm``; chunked gathers as a fallback)

``values`` is usually derived from a much smaller parameter vector (per cell
type, or per connected type pair) with ordinary autograd, e.g.
``values = base * theta[edge_pair]``; that gather is computed once per rollout
chunk and shared across steps, so its cost is not per step.

Layout: the operator works on ``(N, B)`` tensors (neurons first) so CSR
matmuls and row gathers need no transposes. :func:`propagate` accepts the
more common ``(B, N)`` layout.
"""


from dataclasses import dataclass
from typing import Optional

import warnings

import numpy as np
import torch
from torch.autograd.function import once_differentiable

warnings.filterwarnings("ignore", message="Sparse CSR tensor support is in beta state")

DEFAULT_EDGE_CHUNK = 1 << 22  # ~4M edges per chunk in the values gradient


@dataclass
class SparseStructure:
    """Fixed sparsity pattern of the connectome on one device.

    Edges must be sorted by ``(post, pre)``, which is the order
    :class:`meta_brain.graph.ConnectomeArrays` stores them in, so the edge
    order *is* CSR order and ``values`` needs no permutation in the forward.
    """

    n: int
    pre: torch.Tensor  # (E,) long, column index in CSR order
    post: torch.Tensor  # (E,) long, row index in CSR order
    crow: torch.Tensor  # (N+1,) row pointer of A
    t_crow: torch.Tensor  # (N+1,) row pointer of A^T
    t_col: torch.Tensor  # (E,) column index (= post) of A^T
    t_perm: torch.Tensor  # (E,) edge index in A^T order
    edge_chunk: int = DEFAULT_EDGE_CHUNK
    edge_grad_kernel: str = "auto"  # "auto" | "sddmm" | "gather"

    @property
    def e(self) -> int:
        return int(self.pre.numel())

    @property
    def device(self) -> torch.device:
        return self.pre.device

    @classmethod
    def from_edges(cls, pre, post, n: int, device: Optional[torch.device | str] = None,
                   index_dtype: torch.dtype = torch.int64,
                   edge_chunk: int = DEFAULT_EDGE_CHUNK,
                   edge_grad_kernel: str = "auto") -> "SparseStructure":
        pre = np.asarray(pre, dtype=np.int64)
        post = np.asarray(post, dtype=np.int64)
        if len(pre) > 1:
            key = post * n + pre
            if np.any(np.diff(key) <= 0):
                raise ValueError("Edges must be unique and sorted by (post, pre); use graph.sort_edges.")
        if len(pre) and (pre.min() < 0 or post.min() < 0 or max(pre.max(), post.max()) >= n):
            raise ValueError("Edge index out of range.")

        crow = np.zeros(n + 1, dtype=np.int64)
        np.cumsum(np.bincount(post, minlength=n), out=crow[1:])
        t_perm = np.lexsort((post, pre))
        t_crow = np.zeros(n + 1, dtype=np.int64)
        np.cumsum(np.bincount(pre, minlength=n), out=t_crow[1:])

        def to(a: np.ndarray, dtype: torch.dtype = index_dtype) -> torch.Tensor:
            return torch.as_tensor(a).to(device=device, dtype=dtype)

        return cls(n=n, pre=to(pre, torch.long), post=to(post, torch.long), crow=to(crow),
                   t_crow=to(t_crow), t_col=to(post[t_perm]), t_perm=to(t_perm, torch.long),
                   edge_chunk=edge_chunk, edge_grad_kernel=edge_grad_kernel)

    def to(self, device: torch.device | str) -> "SparseStructure":
        return SparseStructure(self.n, *(getattr(self, f).to(device) for f in
                                         ("pre", "post", "crow", "t_crow", "t_col", "t_perm")),
                               edge_chunk=self.edge_chunk, edge_grad_kernel=self.edge_grad_kernel)

    def matrix(self, values: torch.Tensor) -> torch.Tensor:
        col = self.pre.to(self.crow.dtype)
        return torch.sparse_csr_tensor(self.crow, col, values, (self.n, self.n))

    def transpose_matrix(self, values: torch.Tensor) -> torch.Tensor:
        return torch.sparse_csr_tensor(self.t_crow, self.t_col, values[self.t_perm], (self.n, self.n))

    def nbytes(self) -> int:
        return sum(t.numel() * t.element_size() for t in
                   (self.pre, self.post, self.crow, self.t_crow, self.t_col, self.t_perm))


def _edge_grad(structure: SparseStructure, grad_y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """``sum_b grad_y[post_e, b] * x[pre_e, b]`` for every edge, in CSR (= edge) order.

    This is a sampled dense-dense matmul (``grad_y @ x.T`` read only at the
    connectome's nonzeros), which cuSPARSE computes far faster than gathering
    two ``E x B`` tensors.
    """
    kernel = structure.edge_grad_kernel
    if kernel not in ("auto", "sddmm", "gather"):
        raise ValueError(f"unknown edge_grad_kernel {kernel!r}")
    if kernel != "gather":
        try:
            pattern = structure.matrix(torch.zeros(structure.e, dtype=grad_y.dtype, device=grad_y.device))
            return torch.sparse.sampled_addmm(pattern, grad_y, x.t().contiguous(), beta=0.0).values()
        except (RuntimeError, NotImplementedError):
            if kernel == "sddmm":
                raise
    return _edge_grad_gather(structure, grad_y, x)


def _edge_grad_gather(structure: SparseStructure, grad_y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """Fallback: the same edge gradient from chunked row gathers."""
    out = torch.empty(structure.e, dtype=grad_y.dtype, device=grad_y.device)
    step = max(1, int(structure.edge_chunk))
    for s in range(0, structure.e, step):
        e = min(s + step, structure.e)
        out[s:e] = (grad_y.index_select(0, structure.post[s:e]) *
                    x.index_select(0, structure.pre[s:e])).sum(dim=1)
    return out


class _SparsePropagate(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:
        ctx.structure = structure
        ctx.save_for_backward(x, values)
        return torch.sparse.mm(structure.matrix(values), x)

    @staticmethod
    @once_differentiable
    def backward(ctx, grad_y: torch.Tensor):
        x, values = ctx.saved_tensors
        structure: SparseStructure = ctx.structure
        grad_y = grad_y.contiguous()
        grad_x = grad_values = None
        if ctx.needs_input_grad[0]:
            grad_x = torch.sparse.mm(structure.transpose_matrix(values), grad_y)
        if ctx.needs_input_grad[1]:
            grad_values = _edge_grad(structure, grad_y, x)
        return grad_x, grad_values, None


def sparse_propagate(x_nb: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:
    """``A(values) @ x`` for ``x`` of shape ``(N, B)``."""
    if x_nb.dim() != 2 or x_nb.shape[0] != structure.n:
        raise ValueError(f"x must be (N={structure.n}, B), got {tuple(x_nb.shape)}")
    if values.shape != (structure.e,):
        raise ValueError(f"values must be (E={structure.e},), got {tuple(values.shape)}")
    return _SparsePropagate.apply(x_nb.contiguous(), values, structure)


def propagate(x_bn: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:
    """Batch-first convenience wrapper: ``(B, N) -> (B, N)``."""
    return sparse_propagate(x_bn.t(), values, structure).t()


def reference_propagate(x_nb: torch.Tensor, values: torch.Tensor, structure: SparseStructure) -> torch.Tensor:
    """Plain-autograd gather/scatter reference used by the tests and the benchmark."""
    msg = x_nb.index_select(0, structure.pre) * values.unsqueeze(1)
    out = torch.zeros_like(x_nb)
    return out.index_add(0, structure.post, msg)


__all__ = [
    "SparseStructure",
    "sparse_propagate",
    "propagate",
    "reference_propagate",
]


In [ ]:
# --- embedded from meta_brain/benchmark.py ---
"""The milestone-M1 workload: BPTT through the whole connectome.

Shared by ``scripts/benchmark_sparse_propagate.py`` and the Colab notebook.
The workload is the planned layer-1 model: leaky rate dynamics with per-type
``tau`` and ``v0`` and a per-type-pair gain, ``K`` micro-steps per frame,
``T`` frames, batch ``B``, optional per-frame gradient checkpointing.
"""


import gc
import sys
import time
from typing import Callable, Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


GIB = 1024 ** 3
OPS: Dict[str, Callable] = {"custom": sparse_propagate, "reference": reference_propagate}


def synthetic_graph(n: int, e: int, n_types: int, seed: int = 0, n_readout: int = 1300) -> dict:
    """Heavy-tailed random graph with connectome-like degree spread.

    Types are random, so almost every edge gets its own type pair; the real
    connectome has far fewer pairs. Use it for timing, not parameter counts.
    """
    rng = np.random.default_rng(seed)
    w_out = rng.pareto(1.3, n) + 1
    w_in = rng.pareto(1.3, n) + 1
    m = int(e * 1.15)  # oversample to compensate for merged duplicates and self-loops
    pre = rng.choice(n, m, p=w_out / w_out.sum())
    post = rng.choice(n, m, p=w_in / w_in.sum())
    keep = pre != post
    count = rng.geometric(0.15, m).astype(np.float32)
    pre, post, count, _ = merge_duplicates(pre[keep], post[keep], count[keep], n)
    if len(pre) > e:
        idx = np.sort(rng.choice(len(pre), e, replace=False))
        pre, post, count = pre[idx], post[idx], count[idx]
    return dict(n=n, pre=pre, post=post, count=count,
                cell_type=rng.integers(0, n_types, n).astype(np.int32), n_types=n_types,
                sign=np.where(rng.random(n) < 0.3, -1.0, 1.0).astype(np.float32),
                readout=rng.choice(n, min(n_readout, n), replace=False), source="synthetic")


def base_weights(pre: np.ndarray, post: np.ndarray, count: np.ndarray, sign: np.ndarray, n: int) -> np.ndarray:
    """Signed log counts normalized by incoming energy (as in the V2 experiment)."""
    w = np.log1p(count).astype(np.float32)
    energy = np.bincount(post, w * w, n).astype(np.float32)
    w = w / np.sqrt(np.maximum(energy[post], 1e-6))
    return w * sign[pre]


def process_peak_bytes() -> int:
    try:
        import psutil

        info = psutil.Process().memory_info()
        return int(getattr(info, "peak_wset", 0) or getattr(info, "rss", 0))
    except ImportError:
        import resource  # Linux / macOS

        scale = 1 if sys.platform == "darwin" else 1024
        return int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * scale)


def _sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def _is_oom(exc: BaseException) -> bool:
    oom = getattr(torch.cuda, "OutOfMemoryError", MemoryError)
    return isinstance(exc, (oom, MemoryError)) or "out of memory" in str(exc).lower()


def bptt_workload(structure: SparseStructure, base: torch.Tensor, edge_pair: torch.Tensor,
                  n_pairs: int, cell_type: torch.Tensor, n_types: int, readout: torch.Tensor,
                  op: str = "custom", batch: int = 8, frames: int = 4, micro_steps: int = 8,
                  use_checkpoint: bool = True, iters: int = 2, warmup: int = 1, seed: int = 0,
                  dt: float = 0.25, gain: float = 1.6) -> dict:
    """Time forward + backward of the layer-1 workload; returns a result row.

    All tensors must already live on ``structure.device``. On CUDA, peak
    memory is ``torch.cuda.max_memory_allocated`` over the whole call
    (including the graph tensors already resident). An out-of-memory error is
    caught and reported as ``status='OOM'``.
    """
    device = structure.device
    propagate_fn = OPS[op]
    row = {"op": op, "batch": batch, "frames": frames, "micro_steps": micro_steps,
           "checkpoint": bool(use_checkpoint), "neurons": structure.n, "edges": structure.e,
           "type_pairs": int(n_pairs), "cell_types": int(n_types)}
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    try:
        gen = torch.Generator(device="cpu").manual_seed(seed)
        log_tau = torch.zeros(n_types, device=device, requires_grad=True)
        v0 = torch.zeros(n_types, device=device, requires_grad=True)
        pair_gain = torch.zeros(n_pairs, device=device, requires_grad=True)
        params = [log_tau, v0, pair_gain]
        drive = (torch.randn(frames, structure.n, batch, generator=gen) * 0.1).to(device)

        def frame(v: torch.Tensor, values: torch.Tensor, alpha: torch.Tensor, rest: torch.Tensor,
                  ext: torch.Tensor) -> torch.Tensor:
            for _ in range(micro_steps):
                # Saturating rate: with ReLU alone the real connectome's gain-1.6
                # dynamics diverge within ~100 micro-steps (non-finite gradients).
                rec = propagate_fn(torch.tanh(F.relu(v)), values, structure)
                v = v + alpha * (-v + rest + gain * rec + ext)
            return v

        timings = []
        grad_norm = 0.0
        for it in range(warmup + iters):
            _sync(device)
            t_start = time.perf_counter()
            values = base * F.softplus(pair_gain + 0.5413)[edge_pair]  # softplus(0.5413) ~= 1
            alpha = (dt / (1.0 + torch.exp(log_tau)))[cell_type].unsqueeze(1)
            rest = v0[cell_type].unsqueeze(1)
            v = torch.zeros(structure.n, batch, device=device)
            loss = torch.zeros((), device=device)
            for t in range(frames):
                if use_checkpoint:
                    v = checkpoint(frame, v, values, alpha, rest, drive[t], use_reentrant=False)
                else:
                    v = frame(v, values, alpha, rest, drive[t])
                loss = loss + F.relu(v.index_select(0, readout)).square().mean()
            _sync(device)
            t_fwd = time.perf_counter()
            loss.backward()
            _sync(device)
            t_bwd = time.perf_counter()
            grads = [p.grad for p in params]
            if any(g is None or not torch.isfinite(g).all() for g in grads):
                raise FloatingPointError("missing or non-finite gradient")
            grad_norm = float(torch.sqrt(sum((g.double() ** 2).sum() for g in grads)))
            for p in params:
                p.grad = None
            if it >= warmup:
                timings.append((t_fwd - t_start, t_bwd - t_fwd))
            del values, alpha, rest, v, loss
    except (RuntimeError, MemoryError) as exc:
        if not _is_oom(exc):
            raise
        row.update(status="OOM", error=str(exc).splitlines()[0][:200])
        if device.type == "cuda":
            row["peak_memory_gib"] = round(torch.cuda.max_memory_allocated(device) / GIB, 3)
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
        return row

    fwd = float(np.median([a for a, _ in timings]))
    bwd = float(np.median([b for _, b in timings]))
    steps = frames * micro_steps
    peak = (torch.cuda.max_memory_allocated(device) if device.type == "cuda" else process_peak_bytes())
    row.update(
        status="OK",
        forward_s=round(fwd, 4),
        backward_s=round(bwd, 4),
        ms_per_micro_step=round(1000 * (fwd + bwd) / steps, 3),
        grad_norm=grad_norm,
        trainable_params=int(n_types * 2 + n_pairs),
        peak_memory_gib=round(peak / GIB, 3),
        peak_memory_kind="cuda.max_memory_allocated" if device.type == "cuda" else "process peak RSS",
    )
    return row


def prepare_tensors(n: int, pre: np.ndarray, post: np.ndarray, count: np.ndarray, sign: np.ndarray,
                    cell_type: np.ndarray, n_types: int, readout: np.ndarray, device: torch.device | str,
                    edge_pair: Optional[np.ndarray] = None, n_pairs: Optional[int] = None,
                    index_dtype: torch.dtype = torch.int32) -> dict:
    """Move a graph onto ``device`` in the form :func:`bptt_workload` expects."""

    if edge_pair is None:
        edge_pair, pair_types = type_pair_index(pre, post, cell_type, n_types)
        n_pairs = len(pair_types)
    device = torch.device(device)
    structure = SparseStructure.from_edges(pre, post, n, device=device, index_dtype=index_dtype)
    return dict(
        structure=structure,
        base=torch.as_tensor(base_weights(pre, post, count, sign, n), device=device),
        edge_pair=torch.as_tensor(edge_pair, device=device, dtype=torch.long),
        n_pairs=int(n_pairs),
        cell_type=torch.as_tensor(cell_type, device=device, dtype=torch.long),
        n_types=int(n_types),
        readout=torch.as_tensor(readout, device=device, dtype=torch.long),
    )


## 3. M0: carga, cache e auditoria das arestas

A primeira execução monta o grafo a partir dos arquivos e grava o cache. Em seguida o notebook recarrega do cache para medir o tempo do critério. O relatório precisa fechar a conta: lidas = mantidas + descartadas + mescladas.

In [ ]:
t0 = time.perf_counter()
graph = load_connectome(DATA_DIR, CACHE_DIR, min_weight=MIN_WEIGHT, statuses=STATUSES)
first_call_s = time.perf_counter() - t0
built_this_run = not graph.report.from_cache

t0 = time.perf_counter()
graph = load_connectome(DATA_DIR, CACHE_DIR, min_weight=MIN_WEIGHT, statuses=STATUSES, verbose=False)
cache_load_s = time.perf_counter() - t0

r = graph.report
if graph.n < MIN_NEURONS or graph.e < MIN_EDGES:
    raise RuntimeError(f"Full-brain guard failed: {graph.n:,} neurons, {graph.e:,} edges "
                       f"(need {MIN_NEURONS:,} / {MIN_EDGES:,}).")
if r.accounted_rows() != r.edge_rows_read:
    raise AssertionError("edge accounting does not close")

sign = graph.sign()
in_deg = np.bincount(graph.post, minlength=graph.n)
out_deg = np.bincount(graph.pre, minlength=graph.n)
RESULTS["m0_load"] = {
    "report": asdict(r),
    "built_this_run": built_this_run,
    "first_call_seconds": round(first_call_s, 2),
    "cache_load_seconds": round(cache_load_s, 2),
    "neurons": graph.n,
    "edges": graph.e,
    "sensory": int(len(graph.sensory_indices())),
    "motor": int(len(graph.motor_indices())),
    "descending": int(len(graph.descending_indices())),
    "inhibitory_neuron_fraction": round(float((sign < 0).mean()), 4),
    "inhibitory_edge_fraction": round(float((sign[graph.pre] < 0).mean()), 4),
    "mean_in_degree": round(float(in_deg.mean()), 2),
    "max_in_degree": int(in_deg.max()),
    "max_out_degree": int(out_deg.max()),
    "isolated_neurons": int(((in_deg == 0) & (out_deg == 0)).sum()),
}
print(f"\ncache load: {cache_load_s:.1f}s | first call: {first_call_s:.1f}s "
      f"({'built' if built_this_run else 'cached'})")
print(json.dumps({k: v for k, v in RESULTS["m0_load"].items() if k != "report"}, indent=2))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

ax = axes[0, 0]
acct = {
    "retained": r.edges_retained,
    "duplicate rows\nmerged": r.duplicate_rows_merged,
    "unknown pre": r.dropped_unknown_pre,
    "unknown post": r.dropped_unknown_post,
    "self-loops": r.dropped_self_loops,
    f"pair < {r.min_weight:g}": r.dropped_below_min_weight,
}
ax.barh(list(acct)[::-1], np.maximum(list(acct.values())[::-1], 1), color="#4878d0")
ax.set_xscale("log")
ax.set_title(f"Edge rows read: {r.edge_rows_read:,}")
for i, v in enumerate(list(acct.values())[::-1]):
    ax.text(max(v, 1), i, f" {v:,}", va="center", fontsize=8)

ax = axes[0, 1]
for deg, label in ((in_deg, "in-degree"), (out_deg, "out-degree")):
    vals, counts = np.unique(deg[deg > 0], return_counts=True)
    ax.loglog(vals, counts, ".", alpha=0.5, label=label)
ax.set_title("Degree distribution")
ax.set_xlabel("degree")
ax.set_ylabel("neurons")
ax.legend()

ax = axes[0, 2]
bins = np.logspace(0, np.log10(max(graph.count.max(), 10)), 50)
ax.hist(graph.count, bins=bins, color="#6acc64")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Synapses per neuron pair")
ax.set_xlabel("synapses")

ax = axes[1, 0]
nt = pd.Series(r.nt_counts).sort_values()
colors = ["#d65f5f" if DEFAULT_NT_SIGN.get(k, 1) < 0 else "#4878d0" for k in nt.index]
ax.barh(nt.index, nt.values, color=colors)
ax.set_xscale("log")
ax.set_title("Neurotransmitter (red = inhibitory sign)")

ax = axes[1, 1]
sc = pd.Series(graph.superclass_names)[graph.superclass].value_counts().head(12)[::-1]
ax.barh(sc.index, sc.values, color="#956cb4")
ax.set_xscale("log")
ax.set_title("Neurons per superclass")

ax = axes[1, 2]
type_sizes = np.bincount(graph.cell_type)
ax.hist(type_sizes, bins=np.logspace(0, np.log10(type_sizes.max() + 1), 40), color="#ee854a")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title(f"Neurons per cell type ({graph.n_types:,} types, {r.n_type_pairs:,} pairs)")

fig.suptitle(f"MaleCNS v1.0 ({', '.join(STATUSES)}): {graph.n:,} neurons, {graph.e:,} edges")
fig.tight_layout()
fig.savefig(OUT_DIR / "m0_connectome.png", dpi=150)
plt.show()

## 4. M0: controles nulos

Cada controle reembaralha só o que precisa:

| Controle | Mantém | Destrói |
|---|---|---|
| `degree_preserving` | graus de entrada e saída e força de saída de cada neurônio | identidade dos parceiros, tabela tipo a tipo |
| `type_preserving` | tabela de conectividade tipo a tipo e graus por bloco (trocas de arestas dentro do bloco) | fiação individual (ex.: retinotopia) |
| `random_sparse` | N, E e o multiconjunto de pesos | todo o resto |

A reciprocidade e a sobreposição com o grafo original mostram quanto de estrutura cada controle remove.

In [ ]:
def edge_key(pre, post, n):
    return post.astype(np.int64) * n + pre.astype(np.int64)


def fraction_present(keys, sorted_ref):
    i = np.minimum(np.searchsorted(sorted_ref, keys), len(sorted_ref) - 1)
    return float((sorted_ref[i] == keys).mean())


def type_table(pre, post, count):
    ep, pairs = type_pair_index(pre, post, graph.cell_type, graph.n_types)
    key = pairs[:, 0].astype(np.int64) * graph.n_types + pairs[:, 1]
    return key, np.bincount(ep), np.round(np.bincount(ep, count.astype(np.float64)), 3)


ref_keys = edge_key(graph.pre, graph.post, graph.n)  # CSR order: already sorted
ref_table = type_table(graph.pre, graph.post, graph.count)
ref_in, ref_out = in_deg, out_deg
ref_strength = np.bincount(graph.pre, graph.count.astype(np.float64), graph.n)


def describe(name, pre, post, count, report=None, seconds=0.0):
    keys = edge_key(pre, post, graph.n)
    table = type_table(pre, post, count)
    row = {
        "control": name,
        "edges": int(len(pre)),
        "seconds": round(seconds, 1),
        "reciprocity": round(fraction_present(edge_key(post, pre, graph.n), np.sort(keys)), 4),
        "overlap_with_original": round(fraction_present(keys, ref_keys), 4),
        "in_out_degree_preserved": bool(np.array_equal(np.bincount(post, minlength=graph.n), ref_in)
                                        and np.array_equal(np.bincount(pre, minlength=graph.n), ref_out)),
        "out_strength_preserved": bool(np.allclose(np.bincount(pre, count.astype(np.float64), graph.n),
                                                   ref_strength)),
        "type_table_preserved": bool(len(table[0]) == len(ref_table[0])
                                     and all(np.array_equal(a, b) for a, b in zip(table, ref_table))),
    }
    if report is not None:
        row.update(repair_rounds=report.repair_rounds, swap_sweeps=report.swap_sweeps,
                   swaps_accepted=report.swaps_accepted,
                   residual_self_loops_dropped=report.residual_self_loops_dropped,
                   residual_duplicates_merged=report.residual_duplicates_merged,
                   residual_fraction=round((report.residual_self_loops_dropped
                                            + report.residual_duplicates_merged) / report.edges_in, 6))
    return row


rows = [describe("original", graph.pre, graph.post, graph.count)]
controls = {
    "degree_preserving": lambda: degree_preserving_shuffle(graph.pre, graph.post, graph.count, graph.n,
                                                           seed=SHUFFLE_SEED),
    "type_preserving": lambda: type_preserving_shuffle(graph.pre, graph.post, graph.count, graph.n,
                                                       graph.cell_type, seed=SHUFFLE_SEED),
    "random_sparse": lambda: random_sparse_graph(graph.n, graph.e, graph.count, seed=SHUFFLE_SEED),
}
degree_samples = {}
for name, make in controls.items():
    t0 = time.perf_counter()
    (p, q, c), rep = make()
    rows.append(describe(name, p, q, c, rep, time.perf_counter() - t0))
    degree_samples[name] = np.bincount(q, minlength=graph.n)
    print(rows[-1], flush=True)
    del p, q, c
    gc.collect()

shuffles = pd.DataFrame(rows).set_index("control")
shuffles.to_csv(OUT_DIR / "m0_shuffles.csv")
RESULTS["m0_shuffles"] = shuffles.reset_index().to_dict(orient="records")
shuffles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
names = list(shuffles.index)
palette = ["#4c4c4c", "#4878d0", "#6acc64", "#d65f5f"]

axes[0].bar(names, shuffles["reciprocity"], color=palette)
axes[0].set_title("Reciprocity (fraction of edges with a reverse edge)")
axes[1].bar(names, shuffles["overlap_with_original"], color=palette)
axes[1].set_title("Edges shared with the original graph")
for ax in axes[:2]:
    ax.tick_params(axis="x", rotation=20)
    for i, v in enumerate(ax.containers[0].datavalues):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax = axes[2]
rng = np.random.default_rng(0)
idx = rng.choice(graph.n, min(20_000, graph.n), replace=False)
for name, color in (("random_sparse", "#d65f5f"), ("degree_preserving", "#4878d0")):
    ax.scatter(ref_in[idx] + 1, degree_samples[name][idx] + 1, s=3, alpha=0.4, color=color, label=name)
lim = ref_in.max() + 2
ax.plot([1, lim], [1, lim], "k--", lw=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("original in-degree + 1")
ax.set_ylabel("control in-degree + 1")
ax.set_title("In-degree: kept by the shuffle, not by a random graph")
ax.legend(markerscale=4)

fig.tight_layout()
fig.savefig(OUT_DIR / "m0_shuffles.png", dpi=150)
plt.show()
shuffles[["in_out_degree_preserved", "out_strength_preserved", "type_table_preserved"]]

## 5. M1: correção do `SparsePropagate`

1. `gradcheck` em float64 num grafo pequeno, em CPU e, se houver, em CUDA.
2. No **grafo completo**, saída e gradientes do operador customizado contra a referência por autograd (gather/scatter).
3. Memória guardada por chamada: o operador customizado guarda só `x` (N×B) e `values` (E). A referência guarda um tensor E×B.

In [ ]:
checks = {}


def small_structure(device, n=40, e=300, seed=0):
    rng = np.random.default_rng(seed)
    p, q, _, _ = merge_duplicates(rng.integers(0, n, e), rng.integers(0, n, e), np.ones(e, np.float32), n)
    return SparseStructure.from_edges(p, q, n, device=device, edge_chunk=37)


for dev in ["cpu"] + (["cuda"] if DEVICE.type == "cuda" else []):
    s = small_structure(dev)
    x = torch.randn(s.n, 3, dtype=torch.float64, device=dev, requires_grad=True)
    v = torch.randn(s.e, dtype=torch.float64, device=dev, requires_grad=True)
    try:
        checks[f"gradcheck_{dev}"] = bool(torch.autograd.gradcheck(lambda a, b: sparse_propagate(a, b, s), (x, v)))
    except Exception as exc:
        checks[f"gradcheck_{dev}"] = False
        checks[f"gradcheck_{dev}_error"] = f"{type(exc).__name__}: {str(exc)[:200]}"

readout = graph.descending_indices()
if len(readout) == 0:
    readout = graph.motor_indices()
edge_pair, pair_types = graph.type_pair_index()
tensors = prepare_tensors(graph.n, graph.pre, graph.post, graph.count, sign, graph.cell_type,
                          graph.n_types, readout, DEVICE, edge_pair=edge_pair, n_pairs=len(pair_types))
S = tensors["structure"]

torch.manual_seed(0)
xb = torch.randn(S.n, 2, device=DEVICE, requires_grad=True)
vb = tensors["base"].clone().requires_grad_(True)
gy = torch.randn(S.n, 2, device=DEVICE)
outs = {}
for name, fn in (("custom", sparse_propagate), ("reference", reference_propagate)):
    y = fn(xb, vb, S)
    gx, gv = torch.autograd.grad(y, (xb, vb), gy)
    outs[name] = (y.detach(), gx, gv)
    del y


def rel_err(a, b):
    return float((a - b).abs().max() / b.abs().max().clamp_min(1e-12))


checks["full_graph_rel_err"] = {k: rel_err(outs["custom"][i], outs["reference"][i])
                                for i, k in enumerate(("y", "grad_x", "grad_values"))}
checks["full_graph_equivalent"] = all(e < 1e-4 for e in checks["full_graph_rel_err"].values())
del outs, gx, gv


def saved_numel(fn):
    sizes = []
    with torch.autograd.graph.saved_tensors_hooks(lambda t: sizes.append(t.numel()) or t, lambda t: t):
        fn()
    return sum(sizes)


xs = torch.randn(S.n, 8, device=DEVICE, requires_grad=True)
checks["saved_numel_per_call"] = {
    "custom": saved_numel(lambda: sparse_propagate(xs, vb, S)),
    "reference": saved_numel(lambda: reference_propagate(xs, vb, S)),
    "N*B": S.n * 8,
    "E*B": S.e * 8,
}
del xs, xb, vb, gy
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
RESULTS["m1_checks"] = checks
print(json.dumps(checks, indent=2))

## 6. M1: benchmark de BPTT pelo cérebro inteiro

A carga de trabalho é a da camada 1: dinâmica de taxa com vazamento, `tau` e `v0` por tipo, ganho por par de tipos, `K` micro-passos por frame e checkpoint por frame. Três grupos:

- **compare:** operador customizado contra a referência na mesma configuração.
- **batch_scaling:** janela de 16 frames com B de 1 a 16. A linha **B=8** é o critério do M1.
- **no_checkpoint:** quanto a memória cresce com o número de frames sem checkpoint.

Configurações que estouram a memória aparecem como `OOM`, o que também é um resultado.

In [ ]:
if BENCH == "full":
    GRID = [
        ("compare", dict(op="custom", batch=8, frames=4, use_checkpoint=True)),
        ("compare", dict(op="reference", batch=8, frames=4, use_checkpoint=True)),
        *[("batch_scaling", dict(op="custom", batch=b, frames=16, use_checkpoint=True)) for b in (1, 4, 8, 16)],
        *[("no_checkpoint", dict(op="custom", batch=8, frames=f, use_checkpoint=False)) for f in (1, 2, 4)],
        ("no_checkpoint", dict(op="reference", batch=8, frames=2, use_checkpoint=False)),
    ]
    CRITERION = dict(group="batch_scaling", op="custom", batch=8, frames=16)
else:
    GRID = [
        ("compare", dict(op="custom", batch=2, frames=2, use_checkpoint=True)),
        ("compare", dict(op="reference", batch=2, frames=2, use_checkpoint=True)),
        *[("batch_scaling", dict(op="custom", batch=b, frames=2, use_checkpoint=True)) for b in (1, 2)],
        *[("no_checkpoint", dict(op="custom", batch=2, frames=f, use_checkpoint=False)) for f in (1, 2)],
    ]
    CRITERION = dict(group="batch_scaling", op="custom", batch=2, frames=2)

bench_rows = []
for group, cfg in GRID:
    t0 = time.perf_counter()
    row = bptt_workload(**tensors, micro_steps=MICRO_STEPS, iters=2, warmup=1, **cfg)
    row.update(group=group, wall_s=round(time.perf_counter() - t0, 1))
    bench_rows.append(row)
    print({k: row.get(k) for k in ("group", "op", "batch", "frames", "checkpoint", "status",
                                   "ms_per_micro_step", "peak_memory_gib", "wall_s")}, flush=True)

bench = pd.DataFrame(bench_rows)
bench.to_csv(OUT_DIR / "m1_benchmark.csv", index=False)
RESULTS["m1_benchmark"] = bench_rows
cols = ["group", "op", "batch", "frames", "checkpoint", "status", "forward_s", "backward_s",
        "ms_per_micro_step", "peak_memory_gib"]
bench[[c for c in cols if c in bench]]

In [ ]:
ok = bench[bench["status"] == "OK"]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
colors = {"custom": "#4878d0", "reference": "#d65f5f"}

cmp_rows = bench[bench["group"] == "compare"]
for ax, metric, title in ((axes[0], "ms_per_micro_step", "Forward + backward per micro-step (ms)"),
                          (axes[1], "peak_memory_gib", "Peak memory (GiB)")):
    vals = cmp_rows.get(metric, pd.Series(0.0, index=cmp_rows.index)).fillna(0).to_numpy()
    ax.bar(cmp_rows["op"], vals, color=[colors[o] for o in cmp_rows["op"]])
    for i, (v, st) in enumerate(zip(vals, cmp_rows["status"])):
        ax.text(i, v, f"{v:.2f}" if st == "OK" else st, ha="center", va="bottom")
    first = cmp_rows.iloc[0]
    ax.set_title(f"{title}\nB={first['batch']}, {first['frames']} frames, K={MICRO_STEPS}")

ax = axes[2]
bs = ok[(ok["group"] == "batch_scaling")]
ax.plot(bs["batch"], bs["peak_memory_gib"], "o-", color=colors["custom"], label="custom + checkpoint")
ax.axhline(M1_MEMORY_TARGET_GIB, color="k", ls="--", lw=1, label=f"M1 target {M1_MEMORY_TARGET_GIB:g} GiB")
if GPU_GIB:
    ax.axhline(GPU_GIB, color="gray", ls=":", lw=1, label=f"{GPU_NAME} {GPU_GIB} GiB")
ax.set_xlabel("batch")
ax.set_ylabel("peak GiB")
ax.set_title(f"Memory vs batch ({CRITERION['frames']}-frame window)")
ax.legend(fontsize=8)

ax = axes[3]
nc = bench[bench["group"] == "no_checkpoint"]
for op, sub in nc.groupby("op"):
    good = sub[sub["status"] == "OK"]
    ax.plot(good["frames"], good["peak_memory_gib"], "o-", color=colors[op], label=f"{op}, no checkpoint")
    for _, bad in sub[sub["status"] != "OK"].iterrows():
        y = bad.get("peak_memory_gib")
        ax.annotate(bad["status"], (bad["frames"], y if pd.notna(y) else 0), color=colors[op])
ax.set_xlabel("frames in the BPTT window")
ax.set_ylabel("peak GiB")
ax.set_title("Memory growth without checkpointing")
ax.legend(fontsize=8)

memory_note = ("" if DEVICE.type == "cuda"
               else "\nCPU: memory is the process-wide peak, so it does not compare configurations")
fig.suptitle(f"M1 on {GPU_NAME or 'CPU'}: {S.n:,} neurons, {S.e:,} edges, "
             f"{tensors['n_pairs']:,} type-pair parameters{memory_note}")
fig.tight_layout()
fig.savefig(OUT_DIR / "m1_benchmark.png", dpi=150)
plt.show()

## 7. Critérios e artefatos

`m0_m1_results.json` é o registro da evidência. `claim_status` só marca `PASS` quando o critério foi medido e atendido.

In [ ]:
def find_row(df, **kw):
    sel = df
    for k, v in kw.items():
        sel = sel[sel[k] == v]
    return sel.iloc[0].to_dict() if len(sel) else None


controls_only = shuffles.drop(index="original")
m0 = {
    "edge_accounting_closes": r.accounted_rows() == r.edge_rows_read,
    "full_brain_guard": graph.n >= MIN_NEURONS and graph.e >= MIN_EDGES,
    "cache_load_seconds": RESULTS["m0_load"]["cache_load_seconds"],
    "cache_load_under_target": RESULTS["m0_load"]["cache_load_seconds"] < M0_CACHE_LOAD_TARGET_S,
    "degree_shuffle_invariants": bool(shuffles.loc["degree_preserving", "in_out_degree_preserved"]
                                      and shuffles.loc["degree_preserving", "out_strength_preserved"]),
    "type_shuffle_invariants": bool(shuffles.loc["type_preserving", "type_table_preserved"]),
    "max_residual_fraction": float(controls_only["residual_fraction"].max()),
}
m0["controls_exact_enough"] = m0["max_residual_fraction"] <= M0_SHUFFLE_RESIDUAL_MAX
m0_pass = all(v for k, v in m0.items() if isinstance(v, bool))

crit = find_row(bench, **CRITERION)
m1 = {
    "gradcheck": all(v for k, v in checks.items() if k.startswith("gradcheck_") and isinstance(v, bool)),
    "full_graph_equivalent": checks["full_graph_equivalent"],
    "criterion_config": CRITERION | {"micro_steps": MICRO_STEPS},
    "criterion_status": crit and crit["status"],
    "criterion_peak_gib": crit and crit.get("peak_memory_gib"),
    "criterion_ms_per_micro_step": crit and crit.get("ms_per_micro_step"),
}
cmp_ok = bench[(bench["group"] == "compare") & (bench["status"] == "OK")].set_index("op")
if {"custom", "reference"} <= set(cmp_ok.index):
    m1["speedup_vs_reference"] = round(cmp_ok.loc["reference", "ms_per_micro_step"]
                                       / cmp_ok.loc["custom", "ms_per_micro_step"], 2)
    m1["memory_ratio_vs_reference"] = round(cmp_ok.loc["reference", "peak_memory_gib"]
                                            / cmp_ok.loc["custom", "peak_memory_gib"], 2)
m1_fits = bool(crit and crit["status"] == "OK" and crit["peak_memory_gib"] < M1_MEMORY_TARGET_GIB)
if DEVICE.type != "cuda" or BENCH != "full":
    m1_status = "NOT_EVALUATED_NO_CUDA" if DEVICE.type != "cuda" else "NOT_EVALUATED_TINY_GRID"
else:
    m1_status = "PASS" if (m1["gradcheck"] and m1["full_graph_equivalent"] and m1_fits) else "FAIL"

RESULTS["criteria"] = {"m0": m0, "m1": m1}
RESULTS["claim_status"] = {"M0": "PASS" if m0_pass else "FAIL", "M1": m1_status}


def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (Path, torch.device)):
        return str(o)
    raise TypeError(type(o).__name__)


(OUT_DIR / "m0_m1_results.json").write_text(json.dumps(RESULTS, indent=2, default=_jsonable), encoding="utf-8")
archive = OUT_DIR.parent / f"{OUT_DIR.name}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.iterdir()):
        z.write(p, p.name)
if DRIVE_DATA is not None:
    shutil.copy2(archive, DRIVE_DATA.parent / archive.name)

print(json.dumps(RESULTS["claim_status"], indent=2))
print(json.dumps(RESULTS["criteria"], indent=2, default=_jsonable))
print("\noutputs:", *sorted(p.name for p in OUT_DIR.iterdir()), archive.name, sep="\n  ")
try:
    from IPython.display import Image, display
    for name in ("m0_connectome.png", "m0_shuffles.png", "m1_benchmark.png"):
        display(Image(filename=str(OUT_DIR / name)))
except ImportError:
    pass